# ML Platform Architecture

Design and build production ML platforms — feature stores, model registries, serving infrastructure, A/B testing, and ML monitoring.

**Topics:** Feature stores, model registry (MLflow), serving patterns, A/B testing for ML, data drift, model degradation monitoring

## 1. Feature Store Design Patterns

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime
from typing import Optional
import json

@dataclass
class FeatureDefinition:
    """Defines a feature with its computation, type, and serving config."""
    name: str
    entity: str          # e.g., 'user_id', 'item_id'
    dtype: str           # 'float32', 'int64', 'string'
    source: str          # table or stream this comes from
    ttl_seconds: int     # how long the feature value is fresh
    description: str = ''
    tags: list[str] = field(default_factory=list)

# Feature registry
FEATURE_REGISTRY = [
    FeatureDefinition('user_age', 'user_id', 'int64', 'users_table', ttl_seconds=86400*30, tags=['demographics']),
    FeatureDefinition('user_30d_purchase_count', 'user_id', 'int64', 'orders_stream', ttl_seconds=3600, tags=['behavior']),
    FeatureDefinition('user_avg_order_value', 'user_id', 'float32', 'orders_table', ttl_seconds=3600*6, tags=['behavior']),
    FeatureDefinition('item_click_through_rate', 'item_id', 'float32', 'events_stream', ttl_seconds=3600, tags=['item']),
    FeatureDefinition('user_item_affinity', 'user_id,item_id', 'float32', 'ml_embeddings', ttl_seconds=3600*4, tags=['embedding']),
]

# Feature store architecture
architecture = {
    'Offline Store': 'Parquet on S3/GCS — used for training, batch inference',
    'Online Store':  'Redis/DynamoDB — low latency (<5ms) for real-time serving',
    'Feature Pipeline': 'Spark/Flink job that computes and syncs features',
    'Serving API':  'FastAPI endpoint that joins features for model input',
    'Registry':     'Central catalog of feature definitions and lineage',
}

print('Feature Registry:')
print(f'{"Name":<30} {"Entity":<20} {"TTL":<12} {"Source"}')
print('-' * 75)
for f in FEATURE_REGISTRY:
    ttl_hrs = f.ttl_seconds / 3600
    print(f'{f.name:<30} {f.entity:<20} {ttl_hrs:.0f}h{"":<7} {f.source}')
print()
print('Feature store layers:')
for layer, desc in architecture.items():
    print(f'  {layer:<20}: {desc}')

## 2. Model Registry with MLflow

In [ ]:
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

def log_model_to_registry(
    model,
    model_name: str,
    metrics: dict[str, float],
    params: dict,
    artifact_path: str = 'model',
) -> str:
    """Log a trained model with full lineage to MLflow registry."""
    with mlflow.start_run() as run:
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        
        mlflow.sklearn.log_model(
            model,
            artifact_path=artifact_path,
            registered_model_name=model_name,
            input_example={'feature_1': 0.5, 'feature_2': 1.2},
            signature=mlflow.models.infer_signature(
                [[0.5, 1.2]], [0.8]
            ),
        )
        return run.info.run_id

def promote_to_production(model_name: str, version: int):
    """Promote model version to Production stage after validation."""
    client = MlflowClient()
    
    # Archive current production model
    current_prod = client.get_latest_versions(model_name, stages=['Production'])
    for mv in current_prod:
        client.transition_model_version_stage(model_name, mv.version, 'Archived')
    
    # Promote new version
    client.transition_model_version_stage(model_name, str(version), 'Production')
    client.update_model_version(
        model_name, str(version),
        description=f'Promoted to Production at {datetime.now().isoformat()}',
    )

# MLflow model lifecycle stages
stages = [
    ('None', 'Freshly registered, not yet reviewed'),
    ('Staging', 'Passed automated tests, awaiting human approval'),
    ('Production', 'Currently serving live traffic'),
    ('Archived', 'Retired — kept for rollback or audit'),
]
print('MLflow model lifecycle:')
for stage, desc in stages:
    print(f'  {stage:<15}: {desc}')

## 3. ML Serving Architecture — Online vs Batch

In [ ]:
import asyncio
from fastapi import FastAPI
from pydantic import BaseModel
import numpy as np

# Online serving: low latency, one prediction at a time
ONLINE_SERVER_CODE = '''
from fastapi import FastAPI
from contextlib import asynccontextmanager
import mlflow.pyfunc

@asynccontextmanager
async def lifespan(app: FastAPI):
    # Load model once at startup (not per request!)
    app.state.model = mlflow.pyfunc.load_model("models:/my-model/Production")
    yield

app = FastAPI(lifespan=lifespan)

@app.post("/predict")
async def predict(features: dict):
    import pandas as pd
    df = pd.DataFrame([features])
    prediction = app.state.model.predict(df)
    return {"prediction": prediction.tolist()}
'''

# Batch serving: high throughput, offline predictions
BATCH_SERVING_CODE = '''
import mlflow.pyfunc
import pandas as pd
from pathlib import Path

def batch_predict(input_path: str, output_path: str, model_uri: str):
    model = mlflow.pyfunc.load_model(model_uri)
    df = pd.read_parquet(input_path)
    
    # Process in chunks to avoid OOM
    chunk_size = 10_000
    predictions = []
    for i in range(0, len(df), chunk_size):
        chunk = df.iloc[i:i+chunk_size]
        preds = model.predict(chunk)
        predictions.extend(preds)
    
    df["prediction"] = predictions
    df.to_parquet(output_path, index=False)
'''

# When to use each
serving_comparison = [
    ('Latency',    'Online: <100ms', 'Batch: minutes-hours'),
    ('Throughput', 'Online: 100-10K rps', 'Batch: millions/hour'),
    ('Use case',   'Real-time recommendations, fraud detection', 'Daily scoring, report generation'),
    ('Cost',       'Higher (always-on infra)', 'Lower (run only when needed)'),
    ('Complexity', 'Higher (SLA, monitoring, scaling)', 'Lower (simple batch job)'),
]
print('Online vs Batch serving:')
for aspect, online, batch in serving_comparison:
    print(f'  {aspect:<12}: Online={online:<40} Batch={batch}')

## 4. A/B Testing and Traffic Splitting for Models

In [ ]:
import hashlib
import random
from dataclasses import dataclass

@dataclass
class ABTestConfig:
    name: str
    control_model: str    # 'Production' stage in registry
    treatment_model: str  # 'Staging' stage
    treatment_traffic: float = 0.1  # 10% traffic to new model
    metric: str = 'click_through_rate'
    min_samples: int = 10_000  # samples before drawing conclusions

def get_model_assignment(user_id: str, experiment: ABTestConfig) -> str:
    """Deterministic assignment: same user always gets same model.
    
    Determinism is critical: user sees consistent experience,
    and we can attribute metrics to the correct variant.
    """
    hash_val = int(hashlib.md5(f'{experiment.name}:{user_id}'.encode()).hexdigest(), 16)
    bucket = (hash_val % 100) / 100.0
    return 'treatment' if bucket < experiment.treatment_traffic else 'control'

def analyze_ab_test(control_metrics: list[float], treatment_metrics: list[float]) -> dict:
    """Statistical significance test for A/B test results."""
    from scipy import stats
    
    t_stat, p_value = stats.ttest_ind(control_metrics, treatment_metrics)
    effect_size = (np.mean(treatment_metrics) - np.mean(control_metrics)) / np.std(control_metrics)
    
    return {
        'control_mean': np.mean(control_metrics),
        'treatment_mean': np.mean(treatment_metrics),
        'lift_pct': (np.mean(treatment_metrics) / np.mean(control_metrics) - 1) * 100,
        'p_value': p_value,
        'significant': p_value < 0.05,
        'cohens_d': effect_size,
    }

# Simulate A/B test
config = ABTestConfig('rec-model-v3-test', 'model-v2', 'model-v3', treatment_traffic=0.1)

np.random.seed(42)
n_users = 1000
user_ids = [f'user_{i:06d}' for i in range(n_users)]
assignments = [get_model_assignment(uid, config) for uid in user_ids]

control_count = assignments.count('control')
treatment_count = assignments.count('treatment')
print(f'A/B Test: {config.name}')
print(f'Control:   {control_count} users ({control_count/n_users:.1%})')
print(f'Treatment: {treatment_count} users ({treatment_count/n_users:.1%})')
print()

control_metrics = np.random.normal(0.12, 0.02, 5000).tolist()
treatment_metrics = np.random.normal(0.135, 0.02, 500).tolist()
results = analyze_ab_test(control_metrics, treatment_metrics)
print('Results:')
for k, v in results.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

## 5. ML Monitoring — Data Drift and Model Degradation

In [ ]:
import numpy as np
from scipy import stats

def detect_data_drift(
    reference_data: np.ndarray,
    production_data: np.ndarray,
    feature_names: list[str],
    alpha: float = 0.05,
) -> dict[str, dict]:
    """Detect feature distribution shifts using Kolmogorov-Smirnov test."""
    results = {}
    for i, name in enumerate(feature_names):
        ref = reference_data[:, i]
        prod = production_data[:, i]
        
        ks_stat, p_value = stats.ks_2samp(ref, prod)
        psi = compute_psi(ref, prod)  # Population Stability Index
        
        results[name] = {
            'ks_statistic': ks_stat, 'p_value': p_value,
            'drift_detected': p_value < alpha,
            'psi': psi, 'severity': 'HIGH' if psi > 0.2 else 'MEDIUM' if psi > 0.1 else 'LOW',
        }
    return results

def compute_psi(reference: np.ndarray, production: np.ndarray, buckets: int = 10) -> float:
    """PSI: 0=no shift, 0.1=minor, 0.2+=significant retraining needed."""
    breakpoints = np.percentile(reference, np.linspace(0, 100, buckets + 1))
    ref_pct = np.histogram(reference, bins=breakpoints)[0] / len(reference)
    prod_pct = np.histogram(production, bins=breakpoints)[0] / len(production)
    ref_pct = np.where(ref_pct == 0, 1e-6, ref_pct)
    prod_pct = np.where(prod_pct == 0, 1e-6, prod_pct)
    return float(np.sum((prod_pct - ref_pct) * np.log(prod_pct / ref_pct)))

# Simulate drift detection
np.random.seed(42)
feature_names = ['age', 'income', 'credit_score', 'tenure_months']
reference = np.random.randn(10000, 4)
# Simulate drift: age distribution shifted, income very different
production = np.column_stack([
    np.random.randn(1000) + 0.3,   # age: slight shift
    np.random.randn(1000) * 1.8,   # income: large scale change
    np.random.randn(1000),          # credit_score: no drift
    np.random.randn(1000) + 0.1,   # tenure: minor shift
])

drift_results = detect_data_drift(reference, production, feature_names)
print('Data drift detection results:')
print(f'{"Feature":<18} {"PSI":<8} {"Severity":<10} {"Drift?"}')
print('-' * 45)
for feature, result in drift_results.items():
    print(f'{feature:<18} {result["psi"]:<8.3f} {result["severity"]:<10} {"YES" if result["drift_detected"] else "no"}')

## Exercises

1. **Feature store implementation:** Build a minimal feature store with an offline layer (Parquet files) and an online layer (Python dict simulating Redis). Implement `materialize()` to sync offline → online and `get_features()` to retrieve features by entity ID.

2. **Shadow mode deployment:** Implement shadow mode serving where 100% of traffic goes to the production model (responses returned to user), but requests are also silently sent to the new model (response discarded). Compare predictions to catch disagreements before full rollout.

3. **Automated retraining trigger:** Build a monitoring system that: (1) checks PSI daily for each feature, (2) sends a Slack alert when PSI > 0.1, (3) automatically triggers a retraining job (MLflow run) when PSI > 0.2 on 3+ features.